# BIST100 Rejim Tespiti + TCMB Politika Entegrasyonu

## 📊 Proje Hakkında
Bu Colab notebook'u, BIST100, BIST Banka ve BIST Sınai endekslerinin piyasa rejimlerini tespit etmek için **KMeans kümeleme** ve **Gaussian Hidden Markov Models (HMM)** algoritmalarını kullanır. Ayrıca TCMB para politikası kararlarının rejim geçişleri üzerindeki etkisini analiz eder.

### 🎯 Hedefler
- **Piyasa Rejimlerini Sınıflandır**: Risk-On (🟢), Carry Unwind (🔴), Stagflation Sideways (🔵)
- **TCMB Politika Etkisi**: Faiz kararlarının rejim geçişleri üzerindeki etkinliği
- **Güncel Rejim Tahmini**: Anlık rejim ve geçiş olasılıkları
- **Sektörel Performans**: Banka vs. Sınai endekslerinin rejim bazlı getirileri

## ⚠️ Önemli Notlar
- Tüm API'ler **ücretsiz** ve doğrulama içerir
- Çalışma süresi yaklaşık 5-10 dakika
- Veriler son 5 yılı kapsar

---

## 🚀 Kurulum ve Gerekli Kütüphaneler

In [ ]:
# Bazı kütüphaneler Colab'ta önceden yüklü olabilir, ama güvenli bir şekilde yeniden yükleyelim
!pip install -q yfinance pandas numpy scikit-learn hmmlearn matplotlib seaborn plotly requests beautifulsoup4 python-dotenv

import warnings
warnings.filterwarnings('ignore')

## 🔐 API Anahtarları
EVDS API key'i için [evds2.tcmb.gov.tr](https://evds2.tcmb.gov.tr/) adresinden ücretsiz kayıt olun. Aşağıdaki hücreyi çalıştırarak key'i girin.

In [ ]:
from getpass import getpass
import os

# API key'ini güvenli bir şekilde al
EVDS_KEY = getpass("EVDS API Key'i girin: ")

# .env dosyasına kaydet (kişisel kullanım için)
with open('.env', 'w') as f:
    f.write(f"EVDS_KEY={EVDS_KEY}")

## 📥 Ana Kod Dosyasını Yükle

In [ ]:
# GitHub'dan veya yerel dosyadan kod'u yükleyebilirsiniz
# Bu hücrede yerel dosyayı yükleyeceğiz

try:
    from regime_detector import run_analysis, print_current_regime_info, save_figures
    print("✅ Kod dosyası başarıyla yüklendi")
except ImportError as e:
    print(f"❌ Hata: {e}")
    print("📁 Lütfen regime_detector.py dosyasını Colab'a yükleyin")

## 📈 Analizi Çalıştır

In [ ]:
# Analizi çalıştır
results = run_analysis()

## 📊 Görselleştirmeler

In [ ]:
# Ana Rejim Grafiği
results['visualizations']['main_plot'].show()

In [ ]:
# Rejim Bazlı İstatistikler
results['visualizations']['stats_heatmap'].show()

In [ ]:
# Rejim Geçiş Matrisi (HMM)
results['visualizations']['transition_matrix'].show()

In [ ]:
# Sektörel Performans Karşılaştırması
results['visualizations']['sector_performance'].show()

## 🎯 Güncel Rejim Bilgisi

In [ ]:
print_current_regime_info(results['current_regime'])

## 📝 Policy Analysis Detayları

In [ ]:
# TCMB faiz kararları detayları
policy_analysis = results['policy_analysis']

print("\n📊 TCMB Politik Kararları (2020-2025):")
print(policy_analysis[['date', 'policy_rate', 'change_bps', 'change_type', 'prev_regime', 'next_regime', 'transition']].head(10))

print("\n📈 Ortalama Rejim Değişimleri:")
print(policy_analysis.groupby('change_type')['prev_regime'].value_counts(normalize=True).unstack())

print("\n📊 Faiz Değişimi vs. Rejim Geçişi:")
policy_analysis['change_category'] = pd.cut(
    policy_analysis['change_bps'], 
    bins=[-np.inf, -100, -25, 25, 100, np.inf],
    labels=['-100+ bps', '-100-25 bps', '±25 bps', '+25-100 bps', '+100+ bps']
)
print(policy_analysis.groupby('change_category')['transition'].value_counts())

## 💾 Sonuçları Kaydet

In [ ]:
# Görselleştirmeleri HTML dosyaları olarak kaydet
save_figures(results['visualizations'], output_dir='visualizations')

# Features ve analiz sonuçlarını CSV olarak kaydet
results['features'].to_csv('market_features.csv')
results['policy_analysis'].to_csv('policy_analysis.csv')

print("✅ Sonuçlar başarıyla kaydedildi!")
print("- HTML görseller: visualizations/ dizini")
print("- Özellikler: market_features.csv")
print("- Politika analizi: policy_analysis.csv")

# Zip dosyası oluştur (indirme için)
!zip -r results.zip visualizations/ market_features.csv policy_analysis.csv

## 📚 Kaynaklar
- **Fiyat Verileri**: Yahoo Finance (yfinance)
- **TCMB Verileri**: EVDS API
- **Algoritmalar**: Scikit-learn (KMeans), hmmlearn (HMM)
- **Görselleştirme**: Plotly, Seaborn
- **TCMB PPK Kararları**: TCMB resmi yayınları (2020-2025)

## 🔄 Güncelleme Tarihi
Son güncelleme: 2024-03-21